# Entrenamiento en Kaggle (modelos clásicos, sin redes neuronales)

Antes de correr:
1. **Add Input** → agrega el dataset combinado del equipo o, si aún no existe, `moltean/fruits` (Fruits-360).
2. **Settings** → Internet: **On** (requiere teléfono verificado). No hace falta GPU: todo corre en CPU.
3. Para correr sin tener la pestaña abierta: **Save Version → Save & Run All**.

Este notebook solo clona el repo y llama a `src/`. Los cambios de código van en el repo, no aquí.

In [ ]:
import os, subprocess

REPO = "/kaggle/working/repo"
if not os.path.isdir(REPO):
    r = subprocess.run(["git", "clone", "https://github.com/11909/Clasificador-fruta-verdura.git", REPO],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("No se pudo clonar el repo. Activa Settings → Internet en el panel derecho "
                           "(Kaggle pide verificar tu teléfono) y vuelve a correr esta celda.\n" + r.stderr)
else:
    subprocess.run(["git", "-C", REPO, "pull", "-q"])  # trae la última versión de main
%cd {REPO}
!pip install -q -r requirements.txt

Localiza las carpetas `Training` y `Test` del dataset adjunto y genera la config de esta corrida.

In [ ]:
import glob, yaml

import os

# Si está adjunto el dataset combinado del equipo (tiene manifest.csv) se usa ese; si no, Fruits-360.
equipo = glob.glob("/kaggle/input/**/manifest.csv", recursive=True)
if equipo:
    base = os.path.dirname(equipo[0])
    train_dirs, test_dirs = [f"{base}/train"], [f"{base}/test"]
else:
    train_dirs = sorted(glob.glob("/kaggle/input/**/Training", recursive=True))
    test_dirs = sorted(glob.glob("/kaggle/input/**/Test", recursive=True))
if not train_dirs or not test_dirs:
    print("Contenido de /kaggle/input:", glob.glob("/kaggle/input/*/*")[:20])
    raise FileNotFoundError("No encontré las carpetas de imágenes. Adjunta el dataset con Add Input "
                            "(Fruits-360: moltean/fruits, o el dataset combinado del equipo) y vuelve a correr.")
print("train:", train_dirs)
print("test:", test_dirs)

cfg = yaml.safe_load(open("configs/baseline.yaml"))
cfg["data"]["train_dirs"] = train_dirs[:1]   # ajusta si adjuntas más de un dataset
cfg["data"]["test_dirs"] = test_dirs[:1]
cfg["data"]["max_por_clase"] = 60            # súbelo para usar más imágenes (más lento)
cfg["data"]["excluir_grupos"] = []           # ["nueces_semillas"] para sacar nueces y semillas
yaml.safe_dump(cfg, open("configs/run.yaml", "w"), allow_unicode=True)

Revisa que todas las clases tengan etiqueta en `data/mapping.csv`.

In [ ]:
import subprocess
r = subprocess.run(["python", "scripts/check_mapping.py", train_dirs[0]], capture_output=True, text=True)
print("\n".join(r.stdout.splitlines()[-5:]), r.stderr)


Extrae características, compara los modelos con validación cruzada, reentrena el mejor y lo evalúa en test.

In [ ]:
!python -m src.train --config configs/run.yaml

Resultados: comparación de modelos (`resultados_cv.csv`), métricas en test y el modelo guardado (`modelo.joblib`) en `/kaggle/working/outputs/`.

In [ ]:
import csv, json
for fila in csv.DictReader(open("/kaggle/working/outputs/resultados_cv.csv")):
    print(f"{fila['modelo']:20} f1_macro={fila['f1_macro']}  ±{fila['f1_macro_std']}  ({fila['segundos']} s)")
print(json.dumps(json.load(open("/kaggle/working/outputs/test_metrics.json")), indent=2))

## Probar con una foto

Sube tus fotos con **Add Input → Upload Dataset** y pon la ruta abajo. Si no pones ninguna, se usa una imagen al azar de Test.

Para no reentrenar en cada sesión: después de entrenar haz **Save Version → Save & Run All**. En sesiones siguientes agrega esa versión como input (**Add Input → Your Work → este notebook**) y esta celda encontrará el `modelo.joblib` guardado.


In [ ]:
import os, sys, glob, random
REPO = "/kaggle/working/repo"
if not os.path.isdir(REPO):
    raise RuntimeError("La sesión se reinició y se borró /kaggle/working. Corre primero la celda que clona el repo.")
os.chdir(REPO)
sys.path.insert(0, REPO)
from PIL import Image
from src.predict import cargar, predecir

# Usa el modelo recién entrenado; si no existe, busca uno guardado en los inputs
# (por ejemplo, la salida de una versión anterior de este notebook agregada con Add Input).
modelos = glob.glob("/kaggle/working/outputs/modelo.joblib") or glob.glob("/kaggle/input/**/modelo.joblib", recursive=True)
if not modelos:
    raise FileNotFoundError("No hay modelo entrenado. Corre la celda de entrenamiento, o agrega como input "
                            "la salida de una versión guardada de este notebook.")

FOTO = ""   # p. ej. "/kaggle/input/datasets/<usuario>/<dataset>/foto.jpg"
if not FOTO:
    carpetas_test = glob.glob("/kaggle/input/**/Test", recursive=True) + glob.glob("/kaggle/input/**/test", recursive=True)
    FOTO = random.choice(glob.glob(f"{carpetas_test[0]}/*/*.jpg"))

paquete = cargar(modelos[0])
print("modelo:", modelos[0])
print("foto:", FOTO)
print(predecir(paquete, Image.open(FOTO)))
Image.open(FOTO)


Opcional: demo web con un enlace público temporal para probar arrastrando fotos.

In [ ]:
import subprocess
subprocess.run(["pip", "install", "-q", "gradio"])
import gradio as gr
gr.Interface(fn=lambda img: predecir(paquete, img), inputs=gr.Image(type="pil"),
             outputs=gr.Label(), title="¿Fruta o verdura?").launch(share=True)